# Ice-cream quarterly sales: EPA decomposition and forecast

**EPA Method** · a worked example from the StatNex teaching guide

**The story.** An ice-cream maker's quarterly sales for ten years, 2015 to 2024. Sales grow over time, peak every summer, and dropped sharply in 2020. The EPA method is named after Japan's former Economic Planning Agency, which introduced it in the 1960s; StatNex computes it with the standard X-11 moving averages that statistics offices use for "seasonally adjusted" figures.

**The question.** Once we separate the trend from the seasonal pattern, how fast is the business growing, and what should we expect for 2025 to 2027?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `08_time_series/epa_method.csv` (40 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Year / period,Sales ($000)
2015-Q1,3171.0
2015-Q2,4506.0
2015-Q3,5467.0
2015-Q4,3063.0
2016-Q1,3420.0
2016-Q2,4555.0
2016-Q3,5640.0
2016-Q4,3328.0
2017-Q1,3582.0
2017-Q2,4856.0
2017-Q3,6039.0
2017-Q4,3359.0
2018-Q1,3764.0
2018-Q2,5148.0
2018-Q3,6151.0
2018-Q4,3596.0
2019-Q1,3950.0
2019-Q2,5367.0
2019-Q3,6385.0
2019-Q4,3893.0
2020-Q1,3518.0
2020-Q2,5008.0
2020-Q3,5832.0
2020-Q4,3527.0
2021-Q1,4132.0
2021-Q2,5945.0
2021-Q3,7117.0
2021-Q4,4323.0
2022-Q1,4518.0
2022-Q2,6362.0
2022-Q3,7609.0
2022-Q4,4487.0
2023-Q1,4861.0
2023-Q2,6417.0
2023-Q3,7850.0
2023-Q4,4789.0
2024-Q1,4914.0
2024-Q2,6504.0
2024-Q3,8055.0
2024-Q4,4754.0

These cells save `curve_fitting.py`, `epa_method.py`: steps that no library does, written out. The analysis imports them.

In [ ]:
%%writefile curve_fitting.py
"""The thirteen curves of curve fitting, each an ordinary least-squares fit (statsmodels OLS).

Five are polynomials, three are a straight line in a transformed x (1/x, sqrt(x),
log x), and five are made linear by taking logs of y -- the power and exponential
curves, and the three that rise towards a ceiling K. A fit returns the coefficients
as the formula states them, and `model`, the statsmodels regression that was actually
solved: its R-squared, analysis of variance, residuals and prediction intervals all come
from there. For the last five that regression is on the transformed scale, so those
numbers describe the fit on the scale it was made on.
"""

from dataclasses import dataclass
from typing import Callable

import numpy as np
import statsmodels.api as sm


@dataclass
class Fit:
    name: str
    formula: str
    coefficients: dict  # name -> value, in the order the formula writes them
    model: object  # the statsmodels OLS results of the regression solved
    predict: Callable  # x -> y on the original scale
    back: Callable  # (lower, upper) on the fitted scale -> on the scale of y


def unchanged(lower, upper):
    return lower, upper


def ols(target, columns):
    """Least squares of target on the columns plus a constant (the constant last)."""
    design = np.column_stack([*columns, np.ones(len(target))])
    return sm.OLS(target, design).fit()


def polynomial(x, y, degree, name, formula):
    """y = a1 x + ... + ad x^d + b, solved on a centred and scaled x.

    Raw powers of x are nearly parallel columns when x is far from 0 (x^5 of a year is
    about 3e16), and least squares on them loses digits. The fit is taken on
    t = (x - c) / s instead, which spans the same polynomials and gives the same fitted
    values, and the coefficients are converted back to powers of x only for the table.
    """
    centre = x.mean()
    scale = np.abs(x - centre).max() or 1.0
    t = (x - centre) / scale
    model = ols(y, [t**p for p in range(1, degree + 1)])
    gamma = model.params
    # The polynomial in t, constant first, rewritten as a polynomial in x.
    in_t = np.polynomial.Polynomial(np.concatenate([gamma[-1:], gamma[:-1]]))
    in_x = in_t(np.polynomial.Polynomial([-centre / scale, 1.0 / scale]))
    raw = np.zeros(degree + 1)
    raw[: in_x.coef.size] = in_x.coef
    names = ["a"] if degree == 1 else [f"a{p}" for p in range(1, degree + 1)]
    coefficients = dict(zip(names + ["b"], [*raw[1:], raw[0]]))
    return Fit(name, formula, coefficients, model,
               lambda v: in_t((np.asarray(v, float) - centre) / scale), unchanged)


def straight_line_in(column, x, y, name, formula):
    """y = a f(x) + b for a fixed transformation f of x (1/x, sqrt(x), log x)."""
    model = ols(y, [column(x)])
    a, b = model.params
    return Fit(name, formula, {"a": a, "b": b}, model,
               lambda v: a * column(np.asarray(v, float)) + b, unchanged)


def fit_curve(model, x, y, bound):
    """Fit one model, named as the app names it; None when the data cannot take it."""
    if model in ("Line", "Quadratic", "Cubic", "Quartic", "Quintic"):
        degree = ["Line", "Quadratic", "Cubic", "Quartic", "Quintic"].index(model) + 1
        name = ["Line", "Quadratic", "Cubic", "Quartic", "Quintic"][degree - 1]
        formula = "y=ax+b" if degree == 1 else "y=" + "+".join(
            f"a{p}x" + (f"^{p}" if p > 1 else "") for p in range(1, degree + 1)) + "+b"
        return polynomial(x, y, degree, name, formula)
    if model == "Reciprocal":  # reciprocal: needs x != 0
        return None if (x == 0).any() else straight_line_in(
            lambda v: 1.0 / v, x, y, "Reciprocal", "y=a/x+b")
    if model == "Square root":  # square root: needs x >= 0
        return None if (x < 0).any() else straight_line_in(
            np.sqrt, x, y, "Square root", "y=a√x+b")
    if model == "Logarithm":  # logarithm: needs x > 0
        return None if (x <= 0).any() else straight_line_in(
            np.log, x, y, "Logarithm", "y=alogx+b")

    # The rest are not linear in their coefficients. Taking logs makes them so, and
    # the straight line is fitted on that scale.
    K = bound
    if model == "Power":  # power, y = b x^a: log y = log b + a log x
        if (x <= 0).any() or (y <= 0).any():
            return None
        fit = ols(np.log(y), [np.log(x)])
        a, b = fit.params[0], np.exp(fit.params[1])
        return Fit("Power", "y=bx^a", {"a": a, "b": b}, fit,
                   lambda v: b * np.asarray(v, float) ** a,
                   lambda lo, hi: (np.exp(lo), np.exp(hi)))
    if model == "Exponential":  # exponential, y = b a^x: log y = log b + x log a
        if (y <= 0).any():
            return None
        fit = ols(np.log(y), [x])
        a, b = np.exp(fit.params[0]), np.exp(fit.params[1])
        return Fit("Exponential", "y=ba^x", {"a": a, "b": b}, fit,
                   lambda v: b * a ** np.asarray(v, float),
                   lambda lo, hi: (np.exp(lo), np.exp(hi)))
    # The three curves with a ceiling K. Their transformations decrease as y rises, so
    # an interval's two ends swap over on the way back.
    if model == "Modified exponential":  # modified exponential, y = K - b a^x: log(K - y) = log b + x log a
        if (K - y <= 0).any():
            return None
        fit = ols(np.log(K - y), [x])
        a, b = np.exp(fit.params[0]), np.exp(fit.params[1])
        return Fit("Modified exponential", "y=K-ba^x", {"a": a, "b": b}, fit,
                   lambda v: K - b * a ** np.asarray(v, float),
                   lambda lo, hi: (K - np.exp(hi), K - np.exp(lo)))
    if model == "Logistic":  # logistic, y = K / (1 + b exp(-a x)): log(K/y - 1) = log b - a x
        if (y <= 0).any() or (K - y <= 0).any():
            return None
        fit = ols(np.log(K / y - 1.0), [x])
        a, b = -fit.params[0], np.exp(fit.params[1])
        return Fit("Logistic", "y=K/(1+bexp(-ax))", {"a": a, "b": b}, fit,
                   lambda v: K / (1.0 + b * np.exp(-a * np.asarray(v, float))),
                   lambda lo, hi: (K / (1 + np.exp(hi)), K / (1 + np.exp(lo))))
    if model == "Gompertz":  # Gompertz, y = K b^(a^x): log(-log(y/K)) = log(-log b) + x log a
        if (y / K <= 0).any() or (y / K >= 1).any():
            return None
        fit = ols(np.log(-np.log(y / K)), [x])
        a, b = np.exp(fit.params[0]), np.exp(-np.exp(fit.params[1]))
        return Fit("Gompertz", "y=Kb^(a^x)", {"a": a, "b": b}, fit,
                   lambda v: K * b ** (a ** np.asarray(v, float)),
                   lambda lo, hi: (K * np.exp(-np.exp(hi)), K * np.exp(-np.exp(lo))))
    raise ValueError(f"unknown model {model!r}")

In [ ]:
%%writefile epa_method.py
"""The filters of the EPA method's multiplicative decomposition, observed = T x C x S x I.

The app follows the standard X-11 recipe: a 2 x k moving average for a first trend,
3 x 3 and then 3 x 5 moving averages across years for the seasonal factors, and a
Henderson filter for the final trend. statsmodels' seasonal_decompose does only the
first step -- a 2 x k average, then one fixed factor per period -- and leaves the ends of
the trend empty, so the filters are written out.
"""

import numpy as np

#: Henderson trend filters: 5 terms for quarterly data, 13 for monthly. They pass a
#: local cubic unchanged, so the trend follows a turning point instead of rounding it.
HENDERSON = {
    4: np.array([-0.073, 0.294, 0.558, 0.294, -0.073]),
    12: np.array([-0.019, -0.028, 0.0, 0.066, 0.147, 0.214, 0.240,
                  0.214, 0.147, 0.066, 0.0, -0.028, -0.019]),
}


def symmetric_filter(values, weights):
    """A centred weighted average at every point, the ends included.

    Near the ends the weights that would fall outside the series are dropped and the
    rest rescaled to sum to one, so the most recent period still gets a value.
    np.convolve on the series padded with zeros gives the weighted sums; on a padded
    run of ones it gives the weights that fell inside the series, which are what each
    sum is divided by. (The weights are symmetric, so convolving is the same as
    correlating.)
    """
    half = len(weights) // 2
    sums = np.convolve(np.pad(values, half), weights, mode="valid")
    used = np.convolve(np.pad(np.ones(len(values)), half), weights, mode="valid")
    return sums / used


def centred_average(values, period):
    """The 2 x k moving average: k + 1 terms, half weight at each end, so that every
    season appears exactly once in each window."""
    weights = np.ones(period + 1)
    weights[0] = weights[-1] = 0.5
    return symmetric_filter(values, weights / weights.sum())


def seasonal_filter(matrix, weights):
    """Smooth each period's ratios across the years (one row per period).

    The first and last years have no neighbours on one side, so the end values are
    repeated outwards before the moving average is taken.
    """
    half = len(weights) // 2
    out = np.empty_like(matrix)
    for p, row in enumerate(matrix):
        padded = np.concatenate([[row[0]] * half, row, [row[-1]] * half])
        out[p] = np.convolve(padded, weights, mode="valid")
    return out


def normalise_seasonal(matrix):
    """Rescale each year's factors (one column per year) to average 1."""
    return matrix / matrix.mean(axis=0)

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from epa_method import HENDERSON, centred_average, normalise_seasonal, seasonal_filter, symmetric_filter
from curve_fitting import fit_curve

pd.set_option("display.width", 120)

## The data

In [ ]:
# Every value must be positive (the model splits the series by ratios) and there may be
# no blanks: the order of the periods is the data.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
# One column in time order. Each year has the same number of periods.
observed = data["Sales ($000)"].to_numpy(float)
periods = 4
years = len(observed) // periods
# The years only label the rows.
year_labels = [2015 + i for i in range(years)]
next_year = year_labels[-1] + 1
# Where the first value falls in its year; it only numbers the periods.
start_period = 1
offset = (start_period - 1) % periods

## Pass 1: a rough trend, then a rough season

In [ ]:
# Averaging over a whole year cancels the season and leaves the trend-cycle (TC).
# Dividing it out leaves season x irregular; smoothing each period's ratios across the
# years (a 3 x 3 moving average) cancels the irregular and leaves the season.
# The ratios are laid out with one row per period and one column per year.
three_by_three = np.convolve(np.ones(3) / 3, np.ones(3) / 3)
three_by_five = np.convolve(np.ones(3) / 3, np.ones(5) / 5)
henderson = HENDERSON[periods]
trend = centred_average(observed, periods)
ratios = (observed / trend).reshape(years, periods).T
seasonal = normalise_seasonal(seasonal_filter(ratios, three_by_three))

## Pass 2: a sharper trend, a better season

In [ ]:
# With the rough season divided out, a Henderson filter follows the trend closely, and
# the season is estimated again from it with a longer 3 x 5 average.
adjusted = observed / seasonal.T.reshape(-1)
trend = symmetric_filter(adjusted, henderson)
ratios = (observed / trend).reshape(years, periods).T
seasonal = normalise_seasonal(seasonal_filter(ratios, three_by_five))

## The four components

In [ ]:
# TCI is the seasonally adjusted series, TC its trend-cycle, I what is left. S is
# published one year beyond the data: the last year's factors, carried forward.
# How to read them: an S of 1.30 means that period usually sells 30% more than an
# average period of its year; an I near 1 means the period behaved as trend and season
# predict, and an I far from 1 marks a period the smooth trend could not follow.
factors = seasonal.T.reshape(-1)
tci = observed / factors
tc = symmetric_filter(tci, henderson)
irregular = tci / tc
extended = normalise_seasonal(np.concatenate([seasonal, seasonal[:, -1:]], axis=1))
decomposition = pd.DataFrame({
    "year": [label for label in [*year_labels, next_year] for _ in range(periods)],
    "period": [(offset + p) % periods + 1 for _ in range(years + 1) for p in range(periods)],
    "TCSI": np.append(observed, [np.nan] * periods),
    "TCI": np.append(tci, [np.nan] * periods),
    "TC": np.append(tc, [np.nan] * periods),
    "S": extended.T.reshape(-1),
    "I": np.append(irregular, [np.nan] * periods),
})
print(decomposition.to_string(index=False))

## Curves fitted to the trend-cycle

In [ ]:
# Six curves are fitted to TC against 1, 2, 3, ... -- to TC rather than to the observed
# values, so that the curve's shape goes on the trend and not on the seasonal wiggle.
# The same fits as curve fitting, in curve_fitting.py. The ceiling K of the logistic and
# Gompertz curves is the one given, or 1.1 times the largest TC.
bound = None
if bound is None:
    bound = 1.1 * tc.max()
print(f"\nUpper bound K = {bound}")
x = np.arange(1, len(tc) + 1, dtype=float)
fits = []
for model in ["Line", "Square root", "Logarithm", "Exponential", "Logistic", "Gompertz"]:
    fit = fit_curve(model, x, tc, bound)
    if fit is None:
        print(f"{model} cannot be fitted to TC (check K).")
        continue
    fits.append(fit)
# The table's R-squared is each regression's own (fit.model is its statsmodels OLS
# results), so for the exponential, logistic and Gompertz curves it is on the log scale
# they were fitted on.
curves = pd.DataFrame(
    {fit.name: [fit.model.rsquared, fit.coefficients["a"], fit.coefficients["b"]] for fit in fits},
    index=["R-squared", "coefficient a", "constant b"],
)
print(curves.to_string())
# To compare the curves with one another, measure every one on TC's own scale:
# 1 - (sum of squared gaps between TC and the curve) / (sum of squares of TC about its
# mean). All six have two coefficients, so no adjustment for size is needed. A good
# fit to the past is not a good forecast: the curves can still part ways when extended.
r2_on_tc = pd.Series({fit.name: 1 - ((tc - fit.predict(x)) ** 2).sum() / ((tc - tc.mean()) ** 2).sum()
                      for fit in fits})
print("\nR-squared on TC's own scale:")
print(r2_on_tc.to_string())

## Forecast

In [ ]:
horizon = 3
# From the first observed period, so the curves can be set beside the data.
future = np.arange(1, len(tc) + horizon * periods + 1)
labels = [*year_labels, *[year_labels[-1] + k for k in range(1, horizon + 1)]]
position = future - 1
forecast = pd.DataFrame({"year": [labels[c] for c in position // periods],
                         "period": (offset + position % periods) % periods + 1})
# The bare trend-cycle level, with no season.
for fit in fits:
    forecast[fit.name] = fit.predict(future.astype(float))
print()
print(forecast.to_string(index=False))

## What the result says

The EPA method splits each quarter's sales into a trend-cycle (TC, the long-run path), a seasonal index (S) and an irregular part (I, what is left over). In 2024 the seasonal index is 1.31 for the third quarter and 0.77 for the fourth: summer sells about a third more than the trend, and the last quarter almost a quarter less.

With the seasons removed, TC rises from 4012 in the first quarter of 2015 to 6165 at the end of 2024, in \$000. A straight line fitted to TC has a coefficient of 58.1, so the business grows by about \$58k a quarter. Its coefficient of determination, the share of variation explained, is 0.906.

The forecast depends on the curve. For the last quarter of 2027 the exponential curve gives 7116, the line 6826 and the logarithm curve 5804. The line and the exponential curve fit the past about equally well, so plan with a range, not one number.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.